# MedGeemaSOAP — Full Test Set Inference

Compare **Base MedGemma** vs **Fine-tuned Model** on the entire test dataset.

| Setting | Value |
|---|---|
| Base Model | `google/medgemma-1.5-4b-it` |
| Adapter | `sahabajalam/Med_scribe_V2` |
| Loading | `AutoModelForImageTextToText` + `AutoProcessor` + `bfloat16` |
| Prompt | `processor.apply_chat_template()` (Gemma chat format) |
| Test Data | All samples from `clinical_notes_test.jsonl` |

In [ ]:
# 1. Install
!pip install -q transformers accelerate peft rouge_score bert_score tqdm
print("✅ Done")

In [ ]:
# 2. Login
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HUGGING_FACE_HUB_TOKEN"))
    print("✅ Logged in via Kaggle Secrets")
except Exception:
    login()
    print("✅ Logged in")

In [ ]:
# 3. Config & Load Test Data
import torch, json

BASE_MODEL = "google/medgemma-1.5-4b-it"
ADAPTER = "sahabajalam/Med_scribe_V2"

# Test data path - adjust for your environment
# Kaggle:  "/kaggle/input/clinical-data-v2/clinical_notes_test.jsonl"
# Local:   "../data/processed/clinical_notes_test.jsonl"
TEST_DATA_PATH = "/kaggle/input/clinical-data-v2/clinical_notes_test.jsonl"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

try:
    TEST_SAMPLES = load_jsonl(TEST_DATA_PATH)
    print(f"✅ Loaded {len(TEST_SAMPLES)} test samples from {TEST_DATA_PATH}")
    print(f"   Fields: {list(TEST_SAMPLES[0].keys())}")
    print(f"   First input: {TEST_SAMPLES[0]['input'][:80]}...")
except FileNotFoundError:
    print(f"❌ File not found: {TEST_DATA_PATH}")
    print(f"   Please update TEST_DATA_PATH to point to your test data.")
    TEST_SAMPLES = []

## Base Model

In [ ]:
# 4. Load base model
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f"⬇️ Loading base model: {BASE_MODEL}")
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
)
model.eval()
processor = AutoProcessor.from_pretrained(BASE_MODEL)
print(f"✅ Base model loaded: {type(model).__name__}")

In [ ]:
# 5. Generate with base model
from tqdm.auto import tqdm

def generate_soap(model, processor, messy_note, instruction, max_new_tokens=1024):
    """Generate SOAP note using Gemma chat template."""
    messages = [
        {"role": "user", "content": [{"type": "text", "text": f"{instruction}\n\n{messy_note}"}]}
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    return processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

print(f"🔄 Generating base model predictions for {len(TEST_SAMPLES)} samples...")
base_preds = []
for s in tqdm(TEST_SAMPLES, desc="Base Model"):
    pred = generate_soap(model, processor, s["input"], s["instruction"])
    base_preds.append(pred)

print(f"✅ Base model done: {len(base_preds)} predictions generated")
empty_count = sum(1 for p in base_preds if not p.strip())
if empty_count:
    print(f"   ⚠️  {empty_count}/{len(base_preds)} empty predictions (expected from base model)")
avg_len = sum(len(p) for p in base_preds) / len(base_preds)
print(f"   Average length: {avg_len:.0f} chars")

## Finetuned Model

In [ ]:
# 6. Load adapter
from peft import PeftModel

print(f"⬇️ Loading adapter: {ADAPTER}")
finetuned = PeftModel.from_pretrained(model, ADAPTER)
finetuned.eval()
cfg = list(finetuned.peft_config.values())[0]
print(f"✅ Adapter loaded | r={cfg.r}, alpha={cfg.lora_alpha}")

In [ ]:
# 7. Generate with finetuned model
print(f"🔄 Generating finetuned predictions for {len(TEST_SAMPLES)} samples...")
ft_preds = []
for s in tqdm(TEST_SAMPLES, desc="Finetuned Model"):
    pred = generate_soap(finetuned, processor, s["input"], s["instruction"])
    ft_preds.append(pred)

print(f"✅ Finetuned model done: {len(ft_preds)} predictions generated")
empty_count = sum(1 for p in ft_preds if not p.strip())
if empty_count:
    print(f"   ❌ {empty_count}/{len(ft_preds)} empty predictions (NOT expected)")
avg_len = sum(len(p) for p in ft_preds) / len(ft_preds)
print(f"   Average length: {avg_len:.0f} chars")

## Metrics

In [ ]:
# 8. Calculate metrics
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

references = [s["output"] for s in TEST_SAMPLES]
soap_sections = ["SUBJECTIVE", "OBJECTIVE", "ASSESSMENT", "PLAN"]

def compute_metrics(preds, refs):
    safe = [p if p.strip() else "(empty)" for p in preds]
    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rl = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(safe, refs)]
    # BERTScore
    _, _, F1 = bert_score_fn(safe, refs, lang="en", verbose=False)
    # Format compliance
    fmt = [all(sec in p.upper() for sec in soap_sections) for p in preds]
    return {
        "rouge_l": sum(rl) / len(rl),
        "bertscore_f1": F1.mean().item(),
        "format_compliance": sum(fmt) / len(fmt),
        "empty": sum(1 for p in preds if not p.strip()),
    }

base_m = compute_metrics(base_preds, references)
ft_m = compute_metrics(ft_preds, references)

print(f"\n{'='*70}")
print(f"{'METRIC':<25} {'BASE':>12} {'FINETUNED':>12} {'DELTA':>12}")
print(f"{'='*70}")
for key, label in [("rouge_l","ROUGE-L"), ("bertscore_f1","BERTScore F1"), ("format_compliance","Format Compliance")]:
    b, f = base_m[key], ft_m[key]
    if key == "format_compliance":
        print(f"{label:<25} {b:>11.1%} {f:>11.1%} {f-b:>+11.1%}")
    else:
        print(f"{label:<25} {b:>12.4f} {f:>12.4f} {f-b:>+12.4f}")
print(f"{'='*70}")
if base_m["empty"]: print(f"⚠️  Base: {base_m['empty']}/{len(TEST_SAMPLES)} empty (expected)")
if ft_m["empty"]:   print(f"❌ Finetuned: {ft_m['empty']}/{len(TEST_SAMPLES)} empty")

## Side-by-Side Comparison

In [ ]:
# 9. Display sample comparisons (first 3)
NUM_DISPLAY = min(3, len(TEST_SAMPLES))
print(f"Showing first {NUM_DISPLAY} sample comparisons...\n")

for i in range(NUM_DISPLAY):
    s = TEST_SAMPLES[i]
    print(f"\n{'='*80}")
    print(f"  SAMPLE {i+1}")
    print(f"{'='*80}")

    print(f"\n📥 INPUT:\n{s['input']}")

    print(f"\n✅ REFERENCE (first 400 chars):\n{s['output'][:400]}...")

    print(f"\n🔵 BASE MODEL ({len(base_preds[i])} chars):")
    print(base_preds[i][:400] + "..." if len(base_preds[i]) > 400 else base_preds[i] if base_preds[i] else "(empty)")

    print(f"\n🟢 FINETUNED ({len(ft_preds[i])} chars):")
    print(ft_preds[i][:400] + "..." if len(ft_preds[i]) > 400 else ft_preds[i] if ft_preds[i] else "(empty)")

## Save Results

In [ ]:
# 10. Save
from datetime import datetime

results = {
    "timestamp": datetime.now().isoformat(),
    "config": {"base_model": BASE_MODEL, "adapter": ADAPTER, "num_samples": len(TEST_SAMPLES)},
    "base_metrics": base_m,
    "finetuned_metrics": ft_m,
    "examples": [
        {"input": s["input"], "reference": s["output"],
         "base": base_preds[i], "finetuned": ft_preds[i]}
        for i, s in enumerate(TEST_SAMPLES)
    ],
}

with open("comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Saved to comparison_results.json")
print(f"   Base ROUGE-L:     {base_m['rouge_l']:.4f}")
print(f"   Finetuned ROUGE-L: {ft_m['rouge_l']:.4f}")

## Summary Statistics

In [ ]:
# 11. Summary Statistics
print(f"\n{'='*70}")
print(f"{'FINE-TUNING IMPACT SUMMARY':^70}")
print(f"{'='*70}\n")

# Calculate improvements
rouge_improvement = ((ft_m['rouge_l'] - base_m['rouge_l']) / base_m['rouge_l']) * 100
bert_improvement = ((ft_m['bertscore_f1'] - base_m['bertscore_f1']) / base_m['bertscore_f1']) * 100

print(f"📊 Dataset: {len(TEST_SAMPLES)} clinical notes tested\n")

print(f"🎯 Performance Improvements:")
print(f"   • ROUGE-L:     {base_m['rouge_l']:.4f} → {ft_m['rouge_l']:.4f}  (+{rouge_improvement:.1f}%)")
print(f"   • BERTScore:   {base_m['bertscore_f1']:.4f} → {ft_m['bertscore_f1']:.4f}  (+{bert_improvement:.1f}%)")
print(f"   • Format:      {base_m['format_compliance']:.1%} → {ft_m['format_compliance']:.1%}")

print(f"\n📝 Output Quality:")
avg_base_len = sum(len(p) for p in base_preds) / len(base_preds)
avg_ft_len = sum(len(p) for p in ft_preds) / len(ft_preds)
print(f"   • Avg Base Length:      {avg_base_len:.0f} chars")
print(f"   • Avg Finetuned Length: {avg_ft_len:.0f} chars")
print(f"   • Base Empty:           {base_m['empty']}/{len(TEST_SAMPLES)}")
print(f"   • Finetuned Empty:      {ft_m['empty']}/{len(TEST_SAMPLES)}")

print(f"\n{'='*70}")
if rouge_improvement > 20:
    print("✅ EXCELLENT: Fine-tuning shows significant improvement!")
elif rouge_improvement > 10:
    print("✅ GOOD: Fine-tuning shows meaningful improvement.")
else:
    print("⚠️  Improvement is modest. Consider more training or data quality review.")
print(f"{'='*70}")